<a href="https://colab.research.google.com/github/group6aiml-hub/NIDS_AInML/blob/notebooks/notebooks/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1-2 Oppsett og Last inn data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

path = '/content/drive/MyDrive/Datasets/'
train = pd.read_csv(path + 'UNSW_NB15_training-set.csv', encoding='utf-8-sig')
test  = pd.read_csv(path + 'UNSW_NB15_testing-set.csv',  encoding='utf-8-sig')
print(train.shape, test.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(175341, 45) (82332, 45)


In [ ]:
#3 Fjern ID
train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

In [ ]:
# 4. Manglende verdier
print(train.isna().sum().sum(), test.isna().sum().sum())

0 0


In [ ]:
#5. Duplikater
dup = train.duplicated()
print(dup.sum(), round(dup.mean() * 100, 1), '%')
print(train.loc[dup, 'attack_cat'].value_counts())
train = train.drop_duplicates()
print(train.shape)

67601 38.6 %
attack_cat
Generic           35819
Exploits          13549
DoS                8458
Normal             4110
Reconnaissance     2969
Fuzzers            2034
Analysis            406
Backdoor            211
Shellcode            42
Worms                 3
Name: count, dtype: int64
(107740, 44)


In [ ]:
#6 Motstridende rader
feats = train.columns.drop(['label', 'attack_cat']).tolist()
konflikt = train.groupby(feats)['attack_cat'].nunique()
print((konflikt > 1).sum())

1772


In [ ]:
# 7. Redundant FTP-kolonne
print((train['is_ftp_login'] == train['ct_ftp_cmd']).all())
train = train.drop(columns=['is_ftp_login'])
test  = test.drop(columns=['is_ftp_login'])

True


In [ ]:
#8. Negative verdier
print((train.select_dtypes('number') < 0).sum().sum())

0


In [ ]:
# 9. Klassefordeling etter rensing
print(train['attack_cat'].value_counts())

attack_cat
Normal            51890
Exploits          19844
Fuzzers           16150
Reconnaissance     7522
Generic            4181
DoS                3806
Analysis           1594
Backdoor           1535
Shellcode          1091
Worms               127
Name: count, dtype: int64


In [ ]:
# 10. Grupper sjeldne kategorier
min_antall = 100
for c in ['proto', 'service', 'state']:
    tellinger = train[c].value_counts()
    behold = tellinger[tellinger >= min_antall].index
    train[c] = train[c].where(train[c].isin(behold), 'other')
    test[c]  = test[c].where(test[c].isin(behold), 'other')
    print(c, '→', sorted(behold.tolist()) + ['other'])

proto → ['arp', 'ospf', 'sctp', 'tcp', 'udp', 'unas', 'other']
service → ['-', 'dns', 'ftp', 'ftp-data', 'http', 'pop3', 'smtp', 'ssh', 'other']
state → ['CON', 'FIN', 'INT', 'REQ', 'other']


In [ ]:
# 11. Kontrollsjekk
for c in ['proto', 'service', 'state']:
    print(c, '| bare i test:', set(test[c]) - set(train[c]))
print(train.shape, test.shape)

proto | bare i test: set()
service | bare i test: set()
state | bare i test: set()
(107740, 43) (82332, 43)


In [ ]:
# 12. Lagre
train.to_csv(path + 'train_clean.csv', index=False)
test.to_csv(path + 'test_clean.csv',  index=False)
print('Lagret!')

Lagret!
